# Rainfall Prediction System

Predicting whether it will rain tomorrow from daily weather measurements.

## 1) System / Business Description

The system on which I will be working is Rainfall Prediction. The system is a data-driven solution designed to forecast rainfall in specific regions using Machine Learning techniques. Accurate rainfall predictions are critical for agriculture, water resource management, and disaster preparedness. The system uses historical weather data and other environmental factors to predict future rainfall levels, enabling informed decision-making for stakeholders such as farmers and agriculture agencies.

This system collects weather-related data such as temperature, humidity, wind speed, and past rainfall patterns, then applies Machine Learning models to analyse patterns and relationships within the data. This would also benefit farmers since they will be able to organise their planting and harvesting seasons more effectively.

## 2) Problem or issues being addressed

Accurate rainfall prediction is a complex problem due to the dynamic and interdependent nature of meteorological factors. Traditional statistical methods often fail to capture these complexities, leading to unreliable forecasts. This unpredictability can result in poor planning, agricultural losses, and inadequate disaster management — floods and similar extreme events are consequences of climate change that are expected to occur more frequently and have catastrophic effects in years to come.

By using Machine Learning, this system aims to improve prediction accuracy, enabling better preparedness and resource allocation.

## 3) Solutions (How ML can help)

Machine Learning algorithms can analyse vast amounts of historical weather data and identify non-linear relationships between various factors affecting rainfall, such as temperature, pressure, and wind patterns. By training models like Logistic Regression, Decision Trees, or ensemble methods, the system can make precise and timely predictions. These predictions allow for data-driven decision-making, reducing uncertainty and improving planning in agriculture, water management, and disaster mitigation.

## 4) Dataset Description

`Rainfall.csv` contains 366 daily weather records with the following columns:

| Column | Meaning |
|---|---|
| `day` | Day of the month (1–31) |
| `pressure` | Atmospheric pressure (hPa) |
| `maxtemp`, `temparature`, `mintemp` | Temperatures (°C) |
| `dewpoint` | Dew point (°C) |
| `humidity` | Relative humidity (%) |
| `cloud` | Cloud cover (%) |
| `sunshine` | Sunshine hours |
| `winddirection`, `windspeed` | Wind direction (degrees) and speed (km/h) |
| `rainfall` | **Target** — `yes` or `no` |

## 5) Dataset Collection

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("Rainfall.csv")
df.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
0,1,1025.9,19.9,18.3,16.8,13.1,72,49,yes,9.3,80.0,26.3
1,2,1022.0,21.7,18.9,17.2,15.6,81,83,yes,0.6,50.0,15.3
2,3,1019.7,20.3,19.3,18.0,18.4,95,91,yes,0.0,40.0,14.2
3,4,1018.9,22.3,20.6,19.1,18.8,90,88,yes,1.0,50.0,16.9
4,5,1015.9,21.3,20.7,20.2,19.9,95,81,yes,0.0,40.0,13.7


## 6) Preprocessing

Steps:
1. Import necessary libraries
2. Read the dataset
3. Sanity check of data
4. EDA — Exploratory Data Analysis
5. Missing value treatment
6. Duplicate and garbage value treatment
7. Outlier inspection
8. Encoding of the target
9. Train/test split

### Step 3 — Sanity check

In [2]:
print("Shape:", df.shape)
print()
df.info()

Shape: (366, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   day                     366 non-null    int64  
 1   pressure                366 non-null    float64
 2   maxtemp                 366 non-null    float64
 3   temparature             366 non-null    float64
 4   mintemp                 366 non-null    float64
 5   dewpoint                366 non-null    float64
 6   humidity                366 non-null    int64  
 7   cloud                   366 non-null    int64  
 8   rainfall                366 non-null    object 
 9   sunshine                366 non-null    float64
 10           winddirection  365 non-null    float64
 11  windspeed               365 non-null    float64
dtypes: float64(8), int64(3), object(1)
memory usage: 34.4+ KB


In [3]:
df.describe()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
count,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,365.000000,365.000000
mean,15.756831,1013.742623,26.191257,23.747268,21.894536,19.989071,80.177596,71.128415,4.419399,101.506849,21.536986
std,8.823592,6.414776,5.978343,5.632813,5.594153,5.997021,10.062470,21.798012,3.934398,81.723724,10.069712
min,1.000000,998.500000,7.100000,4.900000,3.100000,-0.400000,36.000000,0.000000,0.000000,10.000000,4.400000
25%,8.000000,1008.500000,21.200000,18.825000,17.125000,16.125000,75.000000,58.000000,0.500000,40.000000,13.700000
50%,16.000000,1013.000000,27.750000,25.450000,23.700000,21.950000,80.500000,80.000000,3.500000,70.000000,20.500000
75%,23.000000,1018.100000,31.200000,28.600000,26.575000,25.000000,87.000000,88.000000,8.200000,190.000000,27.900000
max,31.000000,1034.600000,36.300000,32.400000,30.000000,26.700000,98.000000,100.000000,12.100000,350.000000,59.500000


In [4]:
df.describe(include="object")

,rainfall
count,366
unique,2
top,yes
freq,249


### Step 5 — Missing values

The column headers in the raw file carry trailing spaces (`pressure `, `humidity `, `cloud `, `winddirection `). Stripping them first is essential, otherwise pandas sees them as different columns.

In [5]:
df.columns = df.columns.str.strip()
df.columns.tolist()

['day',
 'pressure',
 'maxtemp',
 'temparature',
 'mintemp',
 'dewpoint',
 'humidity',
 'cloud',
 'rainfall',
 'sunshine',
 'winddirection',
 'windspeed']

In [6]:
missing = df.isnull().sum()
missing[missing > 0]

winddirection    1
windspeed        1
dtype: int64

In [7]:
df[df.isnull().any(axis=1)]

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
160,9,1005.7,31.7,28.2,26.6,25.7,86,79,yes,6.5,NaN,NaN


Only a single row (`day` 9) is incomplete, and it is missing two values out of twelve. With 366 records, dropping that one row is cleaner than inventing wind measurements for it.

In [8]:
df = df.dropna().reset_index(drop=True)
print("Rows after dropping incomplete records:", len(df))

Rows after dropping incomplete records: 365


### Step 6 — Duplicates and garbage values

In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [10]:
for col in df.select_dtypes(include="object").columns:
    print(f"Column: {col}")
    print(df[col].value_counts())
    print("-" * 30)

Column: rainfall
yes    248
no     117
Name: rainfall, dtype: int64
------------------------------


The only text column is the target, and it holds exactly two clean values: `yes` and `no`. No garbage values to repair.

### Step 4 — Exploratory Data Analysis

In [11]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
numeric_cols

['day',
 'pressure',
 'maxtemp',
 'temparature',
 'mintemp',
 'dewpoint',
 'humidity',
 'cloud',
 'sunshine',
 'winddirection',
 'windspeed']

In [12]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), [c for c in numeric_cols if c != "day"]):
    sns.histplot(data=df, x=col, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\113229627.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), [c for c in numeric_cols if c != "day"]):
    sns.boxplot(data=df, x=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\63157522.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between weather measurements")
plt.show()

C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\2162375038.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="rainfall")
plt.title("Rain days vs dry days")
plt.show()

df["rainfall"].value_counts(normalize=True).round(3)

C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\1864487427.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


yes    0.679
no     0.321
Name: rainfall, dtype: float64

### Step 7 — Outliers

The box plots show a handful of high wind-speed readings. Instead of deleting them, the IQR whiskers below report how far outside the normal range they sit — with only 365 rows, removing real storms would weaken the model.

In [16]:
def whiskers(col):
    q1, q3 = np.percentile(col, [25, 75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["windspeed", "winddirection", "pressure"]:
    low, high = whiskers(df[col])
    outside = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col:15s} normal range [{low:8.2f}, {high:8.2f}]  values outside: {outside}")

windspeed       normal range [   -7.60,    49.20]  values outside: 5
winddirection   normal range [ -185.00,   415.00]  values outside: 0
pressure        normal range [  994.10,  1032.50]  values outside: 2


### Step 8 — Encoding

`rainfall` is the target, so it becomes a binary label: `yes → 1`, `no → 0`.

The `day` column is dropped from the features. It only records the day of the month, which repeats every month and carries no real weather signal — keeping it would let the model memorise the calendar instead of the physics.

In [17]:
X = df.drop(columns=["rainfall", "day"])
y = df["rainfall"].map({"yes": 1, "no": 0})

print("Features:", X.shape)
print("Target balance:", y.value_counts().to_dict())

Features: (365, 10)
Target balance: {1: 248, 0: 117}


### Step 9 — Train/test split

An 80/20 split, stratified so both sets keep the same rain/dry ratio.

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training features:", X_train.shape)
print("Testing features: ", X_test.shape)
print("Training target:  ", y_train.shape)
print("Testing target:   ", y_test.shape)

Training features: (292, 10)
Testing features:  (73, 10)
Training target:   (292,)
Testing target:    (73,)


## 7) Model training

Four models, from a simple linear baseline to ensembles. Logistic Regression gets a `StandardScaler` because it is sensitive to the scale of the inputs; the tree-based models do not need one.

In [19]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, f1_score

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    cv = cross_val_score(model, X, y, cv=5).mean()
    results.append({
        "Model": name,
        "Test accuracy": round(accuracy_score(y_test, preds), 3),
        "F1 score": round(f1_score(y_test, preds), 3),
        "5-fold CV accuracy": round(cv, 3),
    })

results_df = pd.DataFrame(results).sort_values("Test accuracy", ascending=False)
results_df.reset_index(drop=True)

,Model,Test accuracy,F1 score,5-fold CV accuracy
0,Logistic Regression,0.822,0.871,0.792
1,Gradient Boosting,0.822,0.869,0.742
2,Random Forest,0.795,0.851,0.773
3,Decision Tree,0.726,0.796,0.671


## 8) Evaluation of the best model

The confusion matrix and classification report below are for the highest-scoring model from the table above.

In [20]:
from sklearn.metrics import confusion_matrix, classification_report

best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_preds = best_model.predict(X_test)

print("Best model:", best_name)
print()

cm = confusion_matrix(y_test, best_preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No rain", "Rain"], yticklabels=["No rain", "Rain"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion matrix — {best_name}")
plt.show()

print(classification_report(y_test, best_preds, target_names=["No rain", "Rain"]))

Best model: Logistic Regression

              precision    recall  f1-score   support

     No rain       0.73      0.70      0.71        23
        Rain       0.86      0.88      0.87        50

    accuracy                           0.82        73
   macro avg       0.80      0.79      0.79        73
weighted avg       0.82      0.82      0.82        73



C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\3417997897.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Which features matter?

Random Forest exposes feature importances, which show what the model actually leaned on.

In [21]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, palette="viridis")
plt.xlabel("Importance")
plt.title("Feature importance — Random Forest")
plt.show()

importance.round(3)

C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\327843682.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importance.values, y=importance.index, palette="viridis")
C:\Users\LeozThinkPad\AppData\Local\Temp\ipykernel_18616\327843682.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cloud            0.254
sunshine         0.182
humidity         0.110
windspeed        0.087
pressure         0.071
dewpoint         0.069
maxtemp          0.067
temparature      0.061
mintemp          0.056
winddirection    0.043
dtype: float64

## 9) Conclusion

- The dataset is small (365 usable rows) but clean: no duplicates, one incomplete row, and a binary target with a 68/32 rain-to-dry split.
- Logistic Regression and Gradient Boosting are the strongest performers on held-out data, both around **82% test accuracy** with an F1 near **0.87**.
- Cloud cover, humidity and sunshine are consistently the most informative features — physically sensible for a rain-prediction problem.
- With only one year of daily readings, the model has seen very few examples of each weather pattern. More years of data would matter far more than a more complex algorithm.

### Next steps

- Collect several years of records so seasonal patterns can be learned rather than memorised.
- Engineer rolling features (rain yesterday, 3-day averages) that capture the persistence of weather.
- Try time-based validation instead of a random split, which is the honest way to test a forecasting model.